# C11x · Thêm seed 4–5: A0, A2

Theo `experiments/preregistration.md`, **phụ lục A2**. Train bằng cùng đường code với seed 1–3 để gộp được 5 seed. Arm đọc từ `decision.json` của notebook val (06/11: A0, A2; A8 bị bỏ vì C3 trượt).

Notebook có **hai đợt**, để tiết kiệm compute unit. Đổi biến `PHASE` ở ô cấu hình.

| Đợt | `PHASE` | Runtime | Làm gì | Thời gian [ước lượng] | CU [ước lượng] |
|---|---|---|---|---|---|
| 1 | `"train"` | **A100** | R0 + 4 run, chấm 9 lần | khoảng 4 giờ | khoảng 21 (60 nếu A100 tốn 15 CU/giờ) |
| 2 | `"analyze"` | **L4** | Bootstrap, phân tích theo GT | khoảng 4 giờ | khoảng 7 (20 nếu L4 tốn 5 CU/giờ) |

**Cách chạy:** Run all. Bị ngắt thì Run all lại; phần đã xong tự bỏ qua. Xong đợt 1, notebook tự dừng và nhắc đổi sang đợt 2.

Bootstrap chỉ ghi file khi xong cả head, nên bị ngắt giữa chừng là mất phần của head đó. Để tab mở, máy không ngủ.

**Căn cứ ước lượng:** train 0,56 giờ/run và chấm 337 giây/run/head đo ở `C11_colab.ipynb`. Bootstrap 0,93 giây/vòng trên L4 với 12 run ở `notes/08-c11-execution/t07-c11x-handoff-2026-10-30.md`, giả định tăng tuyến tính theo số run. Phân tích theo GT: 0,42 giờ cho mỗi mức score, ước từ thời gian đo trên val (A100) trong `experiments/2026-11-06-c11x-explore-val/decision.json`. Tốc độ CU là số cộng đồng đo, Google không công bố: đọc tốc độ thật trong Colab.

## 0. Cấu hình

Đường dẫn giống `C11_colab.ipynb`. Code cũ (`scripts/c11`) đọc từ **Drive**; code mới (`scripts/c11x`) lấy từ
**GitHub** (`BRANCH`). Đổi `BRANCH` thành `main` sau khi PR được merge.

In [1]:
from pathlib import Path
import sys, json, time, subprocess

DRIVE      = Path("/content/drive/MyDrive/NCKH")
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"          # code C11 trên Drive (không có trên GitHub)
DATA_DRIVE = DRIVE / "data"
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"
LOCAL      = Path("/content/aitodr")
PROTOCOL, VERSION = "woX", "oc"
RUNS_OLD   = EXP_DRIVE / "runs-woX"                    # 12 run thăm dò của C11 (A0/A1/A8/A2 × seed 1–3)
RUNS_NEW   = EXP_DRIVE / "runs-woX-c11x"               # run mới theo preregistration
BRANCH     = "admin/c11-novelty-plan"                  # đổi thành "main" sau khi merge
GH_DIR     = Path("/content/nckh_gh")
PERGT_DIR  = EXP_DRIVE / "c11x-pergt-val"              # bảng theo GT trên val của 12 run cũ (dùng chung 2 notebook)
GATE_FILE  = EXP_DRIVE / "c11x-h2-gate.json"           # kết quả cổng H2 (preregistration §5)

class StopRun(Exception):
    """Dừng có chủ đích, kèm lý do (Run all dừng ở ô này)."""

def today():
    """Ngày lúc ghi file (không cố định từ đầu phiên như biến TODAY của C11)."""
    return time.strftime("%Y-%m-%d")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"

BRANCH = "main"                                   # đổi thành "main" sau khi merge
PHASE = "analyze"                                     # "train" (A100) rồi "analyze" (L4)                                      # "train" (A100) rồi "analyze" (L4)
if not EXP_DRIVE.exists():
    raise StopRun(f"Không thấy {EXP_DRIVE}. Acc này cần quyền Editor trên thư mục NCKH và một shortcut tên NCKH "
                  "trong My Drive (Shared with me → chuột phải → Organize → Add shortcut → My Drive), rồi Run all lại.")
# Arm và mức score lấy từ quyết định của notebook val (phụ lục A2, mục 4 và 8), không sửa tay.
DECISION_FILE = EXP_DRIVE / "c11x-explore-val" / "decision.json"
if not DECISION_FILE.exists():
    raise StopRun("Chưa có decision.json của notebook val. Chạy C11x_explore_val_colab.ipynb trước.")
DECISION = json.loads(DECISION_FILE.read_text())
if DECISION["verdict"] != "train":
    raise StopRun(f"Notebook val quyết định {DECISION['verdict']}: không train (phụ lục A2, mục 4).")
if not isinstance(DECISION.get("test_scope"), dict):
    raise StopRun(f"decision.json thiếu test_scope: {DECISION.get('test_scope')}")
ARMS = DECISION["arms_if_training"]                    # ["A0", "A8", "A2"], hoặc ["A0", "A2"] nếu C3 trượt
SCORES = DECISION["test_scope"]["scores"]              # [0.25, 0.05], hoặc [0.25] nếu quá ngân sách giờ
NEW_SEEDS, ALL_SEEDS = [4, 5], [1, 2, 3, 4, 5]
DIAG_EVERY = 10                                       # giống seed 1–3
print("arm:", ARMS, "| score theo GT trên test:", SCORES)
RESULTS = EXP_DRIVE / "c11x-seeds1to5-results"
R0_DIR = EXP_DRIVE / "runs-woX-R0"
RESULTS.mkdir(parents=True, exist_ok=True)
assert PHASE in ("train", "analyze")

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
arm: ['A0', 'A2'] | score theo GT trên test: [0.25, 0.05]


In [2]:
!pip -q install "ultralytics==8.4.174" shapely gdown statsmodels
!rm -rf "{GH_DIR}" && git clone -q --depth 1 -b "{BRANCH}" https://github.com/VoDucNhatku/nckh-group7-tiny-obb "{GH_DIR}"
GH_COMMIT = subprocess.check_output(["git", "-C", str(GH_DIR), "rev-parse", "HEAD"]).decode().strip()
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
for p in ("/content/c11", str(GH_DIR / "scripts/c11x")):
    if p not in sys.path:
        sys.path.insert(0, p)
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           c11x_commit=GH_COMMIT)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

sys.path.insert(0, str(GH_DIR / "scripts/rq2"))
prereg_text = (GH_DIR / "experiments" / "preregistration.md").read_text(encoding="utf-8")
# Khóa thật: phụ lục A2 phải nằm trên main, đã điền kết quả val, và BRANCH dùng đúng bản đó (AGENTS.md §1).
subprocess.run(["git", "-C", str(GH_DIR), "fetch", "-q", "--depth", "1", "origin", "main"], check=True)
main_prereg = subprocess.check_output(["git", "-C", str(GH_DIR), "show", "FETCH_HEAD:experiments/preregistration.md"]).decode()
if "### A2 ·" not in main_prereg or "[chưa chạy]" in main_prereg.split("### A2 ·", 1)[1]:
    raise StopRun("Phụ lục A2 chưa có trên main hoặc còn [chưa chạy]. Điền kết quả val vào A2, merge vào main, rồi Run all.")
if prereg_text != main_prereg:
    raise StopRun('preregistration của BRANCH khác bản trên main. Đặt BRANCH = "main" rồi Run all.')
print(subprocess.check_output(["git", "-C", str(GH_DIR), "log", "-1", "--format=preregistration: %H %ci", "--",
                               "experiments/preregistration.md"]).decode())
if PHASE == "train" and "A100" not in str(ENV["gpu"]):
    raise StopRun(f"Đợt train phải dùng A100 như seed 1–3, đang là {ENV['gpu']}.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{'ultralytics': '8.4.174', 'torch': '2.14.1+cu130', 'numpy': '2.4.6', 'python': '3.13.16', 'gpu': 'NVIDIA A100-SXM4-40GB', 'c11x_commit': '5934ecbe6ae950d5a5a3da8395964139fedf4554'}
preregistration: 5934ecbe6ae950d5a5a3da8395964139fedf4554 2026-11-07 06:48:22 +0700



In [4]:
# Test cũ (55, code C11 trên Drive) + test mới. Bỏ test train trên CPU (vài phút); test nặng khác chạy vì có GPU.
def run_pytest(cwd, args):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *args], cwd=str(cwd), capture_output=True, text=True)
    tail = (r.stdout.strip().splitlines() or [r.stderr.strip()])[-1]
    print(cwd, "→", tail)
    if r.returncode != 0:
        print(r.stdout[-6000:], r.stderr[-2000:])
        raise StopRun(f"Test hỏng ở {cwd}. Sửa trước khi chạy tiếp.")
import cv2, shapely
print("cv2", cv2.__version__, "| shapely", shapely.__version__, "| numpy", numpy.__version__)
# test_fastpath kiểm đường chấm của đợt 1 (đã qua khi chạy đợt 1); đợt 2 không gọi đường này.
run_pytest("/content/c11", ["test_stal_arms.py", "test_evalkit.py", "test_data_rq2.py", "test_bootstrap.py"])

run_pytest(GH_DIR / "scripts/c11x", ["tests", "--ignore=tests/test_c11x_train.py", "-p", "no:cacheprovider"])

run_pytest(GH_DIR / "scripts/rq2", ["test_errors_did.py", "test_recall.py", "-p", "no:cacheprovider"])

cv2 4.14.0 | shapely 2.2.0 | numpy 2.4.6
/content/c11 → 48 passed in 38.04s
/content/nckh_gh/scripts/c11x → 51 passed in 16.17s
/content/nckh_gh/scripts/rq2 → 11 passed, 1 warning in 1.69s


## 1. Dữ liệu và dataset YOLO (giống hệt C11)

Giải nén nhãn + ảnh woX từ Drive, dựng lại `yolo_woX` bằng `aitodr_data.build_yolo_dataset` với **cùng `seed=0`**,
rồi so với `split_manifest.json` đã lưu ngày 20/10. Lệch thì dừng: mọi so sánh với 12 run cũ cần đúng split cũ.

In [5]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có trong DATA_DRIVE"
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")

import aitodr_data as ad
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
old = json.loads(next(EXP_DRIVE.glob(f"*-dataset-{PROTOCOL}/split_manifest.json")).read_text())
assert man["splits"] == old["splits"], ("split khác ngày 20/10", man["splits"], old["splits"])
DATA_YAML = DS / "aitodr.yaml"
print("split khớp:", man["splits"])

aitod_wo_xview_trainval_imgs.zip OK
aitod_wo_xview_test_imgs.zip OK
split khớp: {'train': {'n_images': 8426, 'n_gt': 123272, 'n_scene_groups': 7326}, 'val': {'n_images': 936, 'n_gt': 12003, 'n_scene_groups': 815}, 'test': {'n_images': 9261, 'n_gt': 128768, 'n_scene_groups': 8110}}


In [7]:
import inspect, pandas as pd, c11_pipeline as cp
missing = {"project", "diag_every"} - set(inspect.signature(cp.train_arm).parameters)
if missing:
    raise StopRun(f"c11_pipeline.train_arm trên Drive thiếu tham số {missing}")
absent = [f"c11_{a}_s{s}" for a in ARMS for s in (1, 2, 3) if not (RUNS_OLD / f"c11_{a}_s{s}" / "c11_done.json").exists()]
if absent:
    raise StopRun(f"Thiếu run cũ: {absent}")
TEST_IMAGES = DS / "images" / "test"
RUN_DIRS = [RUNS_OLD / cp.run_name(a, s) for s in ALL_SEEDS for a in ARMS]

## 2. Đợt 1: train R0 và seed 4–5, rồi chấm

- Cùng `c11_pipeline.train_arm` như `C11_colab.ipynb` ô #23. Run mới ghi vào `runs-woX`, cạnh seed 1–3.
- R0 (A2 seed 1) ghi riêng, để không đụng run cũ.

In [6]:
if PHASE == "train":
    jobs = [("A2", 1, R0_DIR)] + [(a, s, RUNS_OLD) for s in NEW_SEEDS for a in ARMS]
    for a, s, project in jobs:
        rd = Path(project) / cp.run_name(a, s)
        if (rd / "c11_done.json").exists():
            print("đã xong", rd); continue
        t0 = time.time()
        print(">>> train", a, s, "GPU:", ENV["gpu"], flush=True)
        cp.train_arm(DATA_YAML, a, s, project=project, diag_every=DIAG_EVERY)
        print(rd.name, f"xong sau {(time.time() - t0) / 3600:.2f} giờ", flush=True)
    evals = [(R0_DIR / cp.run_name("A2", 1), "e2e")] + [(RUNS_OLD / cp.run_name(a, s), h) for s in NEW_SEEDS for a in ARMS
                                                        for h in ("e2e", "nms")]
    for rd, head in evals:
        out = rd / f"eval_{head}.json"
        if out.exists():
            continue
        st = cp.predict_and_eval(rd / "weights" / "last.pt", LABELS / "test", TEST_IMAGES, out, head=head,
                                 version=VERSION, exclude_stems=EXCLUDE, save_dota_dir=rd / f"dota_{head}")
        print(rd.name, head, {k: round(st[k], 4) for k in ("AP", "AP_vt", "AP_t", "AP_s")}, flush=True)
    # Kiểm tái lập R0 (phụ lục A2, mục 5)
    rd0, old0 = R0_DIR / cp.run_name("A2", 1), RUNS_OLD / cp.run_name("A2", 1)
    r0 = {"ap_new": json.loads((rd0 / "eval_e2e.json").read_text())["AP"],
          "ap_old": json.loads((old0 / "eval_e2e.json").read_text())["AP"]}
    r0["d_ap"] = r0["ap_new"] - r0["ap_old"]
    if (old0 / "results.csv").exists() and (rd0 / "results.csv").exists():
        new, old = pd.read_csv(rd0 / "results.csv"), pd.read_csv(old0 / "results.csv")
        new.columns, old.columns = new.columns.str.strip(), old.columns.str.strip()
        col = [c for c in new.columns if "mAP50-95" in c][0]
        r0["d_val_ep24"] = float(new[col].iloc[-1] - old[col].iloc[-1])
    else:
        r0["note"] = "thiếu results.csv: không kiểm được mAP50-95 val ở epoch 24, nên tính là không khớp"
    r0["match"] = "d_val_ep24" in r0 and abs(r0["d_ap"]) <= 0.005 and abs(r0["d_val_ep24"]) <= 0.005
    r0["env"] = ENV
    (RESULTS / "r0_check.json").write_text(json.dumps(r0, indent=2))
    print(json.dumps(r0, indent=1))
    raise StopRun("XONG ĐỢT 1. Ngắt runtime A100 để khỏi tốn CU, đổi sang L4, đặt PHASE = \"analyze\", rồi Run all.")

## 3. Đợt 2: bảng 5 seed, P1, bootstrap

P1: A2 − A0, AP_vt, head e2e, **chỉ tính trên seed mới**. "Lặp lại được" khi Δ của cả seed 4 và seed 5 > 0 và trung bình của hai seed này ≥ MEI_vt = 0,006. Hai seed đều > 0 nhưng trung bình < 0,006: "Có hướng nhưng dưới MEI". Còn lại: "Không lặp lại".

Số gộp seed 1–5 (khoảng t, CI theo cảnh, CI gộp) chỉ để mô tả, và chỉ gộp khi R0 khớp.

In [9]:
# Dựng lại head nms của A2 seed 1–3 từ cache gốc 30–31/10. Ngày 07/11 trên Drive mất eval_nms.json và
# evalcache_nms.npz của 8 run cũ (thùng rác trống); chỉ còn evalcache_nms.unsorted.npz (bản gốc, thứ tự ảnh lúc chấm).
import shutil, numpy as np, c11_bootstrap as cb
for s in (1, 2, 3):
    rd = RUNS_OLD / f"c11_A2_s{s}"
    src, dst, js = rd / "evalcache_nms.unsorted.npz", rd / "evalcache_nms.npz", rd / "eval_nms.json"
    if js.exists():
        print("đã có", js); continue
    assert src.exists(), f"không có {src}"
    c = cb.load_cache(src)
    st = cb.stats_fast(cb.prepare_fast(c), np.ones(len(c["stems"]), dtype=np.int64))
    st.update(_reconstructed_from="evalcache_nms.unsorted.npz (chấm 30-31/10)", _reconstructed_on="2026-11-07")
    js.write_text(json.dumps(st, indent=2))
    if not dst.exists():
        shutil.copy2(src, dst)          # ô sau sẽ sắp lại theo thứ tự ảnh của cache e2e
    print(rd.name, {k: round(st[k], 4) for k in ("AP", "AP_vt", "AP_t", "AP_s")})

# Đối chiếu với số ngày 30/10 (notes/08-c11-execution/t07-c11x-handoff-2026-10-30.md, mục 2)
a2 = [json.loads((RUNS_OLD / f"c11_A2_s{s}" / "eval_nms.json").read_text()) for s in (1, 2, 3)]
a0 = [json.loads((RUNS_OLD / f"c11_A0_s{s}" / "eval_nms.json").read_text()) for s in (1, 2, 3)]
ap = np.mean([d["AP"] for d in a2]); dvt = np.mean([d["AP_vt"] for d in a2]) - np.mean([d["AP_vt"] for d in a0])
print(f"AP nms A2 (TB s1–3) = {ap:.4f}  | sổ 30/10: 0.1366")
print(f"ΔAP_vt nms A2−A0    = {dvt:+.4f} | sổ 30/10: +0.0071")
assert abs(ap - 0.1366) < 2e-4 and abs(dvt - 0.0071) < 2e-4, "lệch số 30/10: dừng, gửi output này"


c11_A2_s1 {'AP': 0.1364, 'AP_vt': 0.0468, 'AP_t': 0.1274, 'AP_s': 0.2055}
c11_A2_s2 {'AP': 0.1395, 'AP_vt': 0.0507, 'AP_t': 0.1271, 'AP_s': 0.2092}
c11_A2_s3 {'AP': 0.1339, 'AP_vt': 0.0473, 'AP_t': 0.1233, 'AP_s': 0.1947}
AP nms A2 (TB s1–3) = 0.1366  | sổ 30/10: 0.1366
ΔAP_vt nms A2−A0    = +0.0071 | sổ 30/10: +0.0071


In [10]:
import c11x_results as cr, c11x_cache as cc, c11_bootstrap as cb
from aitodr_data import scene_key
r0 = json.loads((RESULTS / "r0_check.json").read_text())
print("R0 khớp:", r0["match"], "| Δ AP:", round(r0["d_ap"], 5), "| Δ val:", r0.get("d_val_ep24"))
if not r0["match"]:
    print("CẢNH BÁO: R0 không khớp. Theo phụ lục A2 mục 5, KHÔNG gộp seed 1–3 với seed 4–5 trong bài.")
absent = [rd.name for rd in RUN_DIRS if not (rd / "c11_done.json").exists()]
if absent:
    raise StopRun(f"Chưa train xong: {absent}. Chạy đợt 1 trước.")
print("sắp lại cache:", cc.align_caches(RUN_DIRS, RUNS_OLD / "c11_A0_s1" / "evalcache_e2e.npz"))
res = cr.collect_runs(RUNS_OLD)
res = res[res["arm"].isin(ARMS) & res["seed"].isin(ALL_SEEDS)]
assert (res.groupby(["head", "arm"]).size() == len(ALL_SEEDS)).all(), res.groupby(["head", "arm"]).size()
display(cr.summary_table(res).round(4))
for h in ("e2e", "nms"):
    display(cr.per_seed_table(res, h).round(4))
res.to_csv(RESULTS / "results_long.csv", index=False)
PAIRS = [p for p in [("A2", "A0"), ("A8", "A0"), ("A2", "A8")] if set(p) <= set(ARMS)]
KEYS = ("AP_vt", "AP_t", "AP_s", "AP_m", "AP")
rows = []
for head, B in (("e2e", 10_000), ("nms", 2_000)):
    f = RESULTS / f"bootstrap_{head}_B{B}.json"
    t0 = time.time()
    boot = json.loads(f.read_text()) if f.exists() else cr.run_scene_bootstrap(cb, [RUNS_OLD], head, PAIRS, KEYS, B, scene_key, f)
    print(head, f"{(time.time() - t0) / 3600:.2f} giờ", boot.get("_c11x_meta", {}).get("seeds"), flush=True)
    for x, y in PAIRS:
        for m in KEYS:
            sc, sd = boot[f"{x}-{y}"][m], cr.paired_seed_deltas(res, head, x, y, m)
            assert abs(sc["delta"] - sd["mean"]) < 2e-4, ("Δ từ cache lệch eval json", head, x, y, m, sc["delta"], sd["mean"])
            cmb = cr.combine_scene_seed(sc, sd)
            rows.append(dict(head=head, pair=f"{x}-{y}", metric=m, delta=sc["delta"], scene_lo=sc["ci_lo"], scene_hi=sc["ci_hi"],
                             seed_t_lo=sd.get("t_lo"), seed_t_hi=sd.get("t_hi"), seed_min=sd.get("min"), seed_max=sd.get("max"),
                             s4=sd["per_seed"].get(4), s5=sd["per_seed"].get(5), comb_lo=cmb["lo"], comb_hi=cmb["hi"],
                             comb_excl0=cmb.get("excludes_zero")))
ci = pd.DataFrame(rows); display(ci.round(4)); ci.to_csv(RESULTS / "ci_scene_seed.csv", index=False)
p1 = ci[(ci["head"] == "e2e") & (ci["pair"] == "A2-A0") & (ci["metric"] == "AP_vt")].iloc[0]
MEI_VT = 0.006
new_pos = p1["s4"] > 0 and p1["s5"] > 0
new_mean = (p1["s4"] + p1["s5"]) / 2
verdict = "Lặp lại được" if new_pos and new_mean >= MEI_VT else "Có hướng nhưng dưới MEI" if new_pos else "Không lặp lại"
p1_out = dict(verdict=verdict, s4=p1["s4"], s5=p1["s5"], mean_s4_s5=new_mean, mei=MEI_VT, r0_match=r0["match"],
              descriptive_seeds_1to5=dict(delta=p1["delta"], seed_t=[p1["seed_t_lo"], p1["seed_t_hi"]],
                                          scene_ci=[p1["scene_lo"], p1["scene_hi"]], combined_ci=[p1["comb_lo"], p1["comb_hi"]],
                                          pooled_allowed=r0["match"]))
(RESULTS / "p1_verdict.json").write_text(json.dumps(p1_out, indent=2, ensure_ascii=False, default=float))
print(json.dumps(p1_out, indent=1, ensure_ascii=False, default=float))

R0 khớp: False | Δ AP: 0.0015 | Δ val: -0.018589999999999995
CẢNH BÁO: R0 không khớp. Theo phụ lục A2 mục 5, KHÔNG gộp seed 1–3 với seed 4–5 trong bài.
sắp lại cache: ['c11_A2_s1/evalcache_nms.npz', 'c11_A2_s2/evalcache_nms.npz', 'c11_A2_s3/evalcache_nms.npz']


AP                  AP50                 AP_vt                \
            mean     std count    mean     std count    mean     std count   
head arm                                                                     
e2e  A0   0.1300  0.0013     5  0.3886  0.0031     5  0.0418  0.0013     5   
     A2   0.1336  0.0039     5  0.4035  0.0084     5  0.0506  0.0045     5   
nms  A0   0.1347  0.0026     5  0.3994  0.0067     5  0.0412  0.0019     5   
     A2   0.1360  0.0036     5  0.4094  0.0075     5  0.0487  0.0017     5   

            AP_t                  AP_s                AR1500                
            mean     std count    mean     std count    mean     std count  
head arm                                                                    
e2e  A0   0.1180  0.0021     5  0.2059  0.0050     5  0.2759  0.0021     5  
     A2   0.1243  0.0035     5  0.2028  0.0061     5  0.2822  0.0048     5  
nms  A0   0.1205  0.0028     5  0.2107  0.0055     5  0.2376  0.0026     5  
     A2   0.1252  0.0032     5  0.2036  0.0060     5  0.2395  0.0034     5

AP            AP_s            AP_t           AP_vt        
arm       A0      A2      A0      A2      A0      A2      A0      A2
seed                                                                
1     0.1312  0.1321  0.2135  0.2022  0.1170  0.1242  0.0411  0.0456
2     0.1301  0.1384  0.2016  0.2125  0.1201  0.1275  0.0415  0.0570
3     0.1278  0.1292  0.2083  0.1958  0.1150  0.1192  0.0401  0.0472
4     0.1306  0.1370  0.2036  0.2026  0.1200  0.1276  0.0430  0.0515
5     0.1301  0.1315  0.2024  0.2006  0.1179  0.1227  0.0430  0.0518

AP            AP_s            AP_t           AP_vt        
arm       A0      A2      A0      A2      A0      A2      A0      A2
seed                                                                
1     0.1383  0.1364  0.2205  0.2055  0.1225  0.1274  0.0433  0.0468
2     0.1364  0.1395  0.2081  0.2092  0.1239  0.1271  0.0420  0.0507
3     0.1322  0.1339  0.2093  0.1947  0.1175  0.1233  0.0383  0.0473
4     0.1339  0.1391  0.2080  0.2079  0.1207  0.1275  0.0418  0.0499
5     0.1326  0.1310  0.2076  0.2004  0.1179  0.1204  0.0406  0.0487

[bootstrap] 1000/10000  609s, còn ~5477s
[bootstrap] 2000/10000  1217s, còn ~4870s
[bootstrap] 3000/10000  1826s, còn ~4261s
[bootstrap] 4000/10000  2435s, còn ~3653s
[bootstrap] 5000/10000  3044s, còn ~3044s
[bootstrap] 6000/10000  3652s, còn ~2435s
[bootstrap] 7000/10000  4260s, còn ~1826s
[bootstrap] 8000/10000  4869s, còn ~1217s
[bootstrap] 9000/10000  5477s, còn ~609s
[bootstrap] 10000/10000  6085s, còn ~0s
e2e 1.70 giờ {'A0': [1, 2, 3, 4, 5], 'A1': [1, 2, 3], 'A2': [1, 2, 3, 4, 5], 'A8': [1, 2, 3]}
[bootstrap] 200/2000  104s, còn ~938s
[bootstrap] 400/2000  208s, còn ~833s
[bootstrap] 600/2000  312s, còn ~729s
[bootstrap] 800/2000  417s, còn ~625s
[bootstrap] 1000/2000  521s, còn ~521s
[bootstrap] 1200/2000  625s, còn ~417s
[bootstrap] 1400/2000  729s, còn ~313s
[bootstrap] 1600/2000  834s, còn ~208s
[bootstrap] 1800/2000  938s, còn ~104s
[bootstrap] 2000/2000  1042s, còn ~0s
nms 0.29 giờ {'A0': [1, 2, 3, 4, 5], 'A1': [1], 'A2': [1, 2, 3, 4, 5]}


,head,pair,metric,delta,scene_lo,scene_hi,seed_t_lo,seed_t_hi,seed_min,seed_max,s4,s5,comb_lo,comb_hi,comb_excl0
0,e2e,A2-A0,AP_vt,0.0089,0.0052,0.0145,0.0038,0.0139,0.0045,0.0155,0.0085,0.0087,0.0006,0.0171,True
1,e2e,A2-A0,AP_t,0.0062,0.0043,0.0080,0.0043,0.0082,0.0042,0.0077,0.0077,0.0048,0.0030,0.0095,True
2,e2e,A2-A0,AP_s,-0.0031,-0.0073,0.0008,-0.0148,0.0086,-0.0125,0.0109,-0.0009,-0.0018,-0.0162,0.0099,False
3,e2e,A2-A0,AP_m,-0.0083,-0.0142,-0.0031,-0.0145,-0.0020,-0.0138,-0.0028,-0.0028,-0.0138,-0.0183,0.0018,False
4,e2e,A2-A0,AP,0.0037,0.0020,0.0055,-0.0005,0.0079,0.0009,0.0083,0.0063,0.0014,-0.0012,0.0086,False
5,nms,A2-A0,AP_vt,0.0075,0.0041,0.0144,0.0047,0.0102,0.0035,0.0090,0.0081,0.0081,-0.0004,0.0153,False
6,nms,A2-A0,AP_t,0.0046,0.0022,0.0068,0.0024,0.0069,0.0025,0.0067,0.0067,0.0025,0.0007,0.0086,True
7,nms,A2-A0,AP_s,-0.0071,-0.0107,-0.0022,-0.0166,0.0024,-0.0150,0.0011,-0.0001,-0.0072,-0.0184,0.0041,False
8,nms,A2-A0,AP_m,-0.0085,-0.0154,-0.0025,-0.0143,-0.0026,-0.0158,-0.0045,-0.0045,-0.0072,-0.0193,0.0024,False
9,nms,A2-A0,AP,0.0013,-0.0007,0.0033,-0.0025,0.0051,-0.0018,0.0052,0.0052,-0.0016,-0.0034,0.0060,False


{
 "verdict": "Lặp lại được",
 "s4": 0.008508,
 "s5": 0.008728,
 "mean_s4_s5": 0.008618,
 "mei": 0.006,
 "r0_match": false,
 "descriptive_seeds_1to5": {
  "delta": 0.008856049828096284,
  "seed_t": [
   0.003811704744893776,
   0.013900398583582373
  ],
  "scene_ci": [
   0.00524017704680297,
   0.014474774411391491
  ],
  "combined_ci": [
   0.0005960785602883729,
   0.017116021095904196
  ],
  "pooled_allowed": false
 }
}


## 4. Đợt 2: theo từng GT trên tập test, seed 1–5, head e2e (M1, M2, M3)

- `score ≥ 0,25` là chính; `score ≥ 0,05` kiểm độ nhạy, chỉ chạy nếu notebook val ước đủ ngân sách giờ (`SCORES`). Ở 0,25, "bỏ sót" gồm cả hộp đúng chỗ nhưng điểm thấp.
- Mỗi run lưu một file trên Drive; chạy lại thì bỏ qua run đã xong.

In [11]:
import rq2_errors as re_, rq2_did as did
RQ2_DIR, MECH = GH_DIR / "scripts/rq2", RESULTS / "mechanism"
OFF = MECH / "rq2_test"
if not (OFF / "rq2_per_gt.csv.gz").exists():
    subprocess.run([sys.executable, "rq2_offline.py", "--labels", str(LABELS / "test"), "--out", str(OFF),
                    "--protocol", "woX", "--arms", "A0,A1,A8,A2"], cwd=str(RQ2_DIR), check=True)
off = pd.read_csv(OFF / "rq2_per_gt.csv.gz")
stems = sorted(off.stem.unique())
if set(stems) & set(EXCLUDE):
    raise StopRun("bảng theo GT còn ảnh xView.")
pairs = pd.DataFrame({"a": off.scene, "b": off.stem.map(scene_key)}).drop_duplicates()
if not (len(pairs) == pairs.a.nunique() == pairs.b.nunique()):
    raise StopRun(f"scene_key lệch: {len(pairs)} cặp, {pairs.a.nunique()} và {pairs.b.nunique()} nhóm.")
print(len(off), "GT;", off.scene.nunique(), "cụm cảnh, khớp aitodr_data.scene_key")
off2 = off[["stem", "gt", "scene", "bin", "nn16", "A0_cand", "cls"]].assign(starved_A0=off.A0_cand == 0,
                                                                            density=re_.density_bin(off.nn16).values)

def fates(score):
    d = MECH / f"fates_e2e_score{score:g}"
    d.mkdir(parents=True, exist_ok=True)
    frames = []
    for s in ALL_SEEDS:
        for a in ARMS:
            f = d / f"{a}_s{s}.parquet"
            if not f.exists():
                t0 = time.time()
                re_.fates_run(LABELS / "test", stems, RUNS_OLD / cp.run_name(a, s) / "dota_e2e", score).to_parquet(f)
                print(score, a, s, f"{time.time() - t0:.0f}s", flush=True)
            frames.append(pd.read_parquet(f).assign(arm=a, seed=s))
    df = pd.concat(frames, ignore_index=True).merge(off2, on=["stem", "gt"], how="left", validate="many_to_one")
    assert df["scene"].notna().all(), "có GT không khớp bảng nhãn"
    return df.assign(detected=lambda x: x["det_0.5"])

summary = {"p1": p1_out}
for score in SCORES:
    df, tag = fates(score), f"e2e_score{score:g}"
    t, c = re_.fate_table(df), re_.recall_curve(df)
    t.to_csv(MECH / f"fate_by_arm_bin_{tag}.csv"); c.to_csv(MECH / f"recall_iou_by_arm_bin_{tag}.csv")
    re_.fate_table(df, by=("arm", "bin", "starved_A0")).to_csv(MECH / f"fate_by_arm_bin_starved_{tag}.csv")
    re_.fate_table(df, by=("arm", "bin", "density")).to_csv(MECH / f"fate_by_arm_bin_density_{tag}.csv")
    display(t.round(3)); display(c.round(3))
    vt = df[df.bin == "vt"]
    mech = {"M1_A2-A0_starved_vs_not": did.did_bootstrap(vt, "A2", "A0", group="starved_A0", g1=True, g0=False)}
    if "A8" in ARMS:
        mech["M2_A2-A8_dense_vs_isolated"] = did.did_bootstrap(vt, "A2", "A8", group="density", g1=">=3", g0="isolated")
    for k, r in mech.items():
        r["combined"] = cr.combine_scene_seed(r["scene"], r["seed"])
        print(tag, k, {x: round(r["combined"][x], 4) for x in ("delta", "lo", "hi")}, "CI gộp loại 0:", r["combined"]["excludes_zero"],
              "theo seed:", {s: round(v, 4) for s, v in r["seed"]["per_seed"].items()})
    (MECH / f"mech_{tag}.json").write_text(json.dumps(mech, indent=2, default=float))
    summary[tag] = {k: dict(delta=r["combined"]["delta"], lo=r["combined"]["lo"], hi=r["combined"]["hi"],
                            supported=bool(r["combined"]["delta"] > 0 and r["combined"]["excludes_zero"])) for k, r in mech.items()}
(RESULTS / "env.json").write_text(json.dumps(ENV, indent=2))
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=float))
print(json.dumps(summary, indent=1, ensure_ascii=False, default=float))
print("XONG. Gửi lại ô này và thư mục", RESULTS)

128769 GT; 8107 cụm cảnh, khớp aitodr_data.scene_key
0.25 A0 1 137s
0.25 A2 1 147s
0.25 A0 2 140s
0.25 A2 2 147s
0.25 A0 3 141s
0.25 A2 3 148s
0.25 A0 4 138s
0.25 A2 4 144s
0.25 A0 5 138s
0.25 A2 5 145s


fate    detected        lost_to_neighbour           cls         loc         \
            mean    std              mean    std   mean  std   mean    std   
arm bin                                                                      
A0  l      0.281  0.042             0.000  0.000  0.000  0.0  0.074  0.059   
    m      0.773  0.005             0.002  0.000  0.001  0.0  0.056  0.004   
    s      0.693  0.005             0.004  0.001  0.003  0.0  0.104  0.004   
    t      0.539  0.004             0.002  0.000  0.002  0.0  0.189  0.003   
    vt     0.248  0.006             0.000  0.000  0.001  0.0  0.275  0.005   
A2  l      0.230  0.048             0.000  0.000  0.000  0.0  0.067  0.017   
    m      0.758  0.008             0.002  0.001  0.001  0.0  0.060  0.003   
    s      0.692  0.003             0.004  0.000  0.003  0.0  0.110  0.003   
    t      0.570  0.005             0.002  0.000  0.002  0.0  0.187  0.005   
    vt     0.302  0.005             0.000  0.000  0.001  0.0  0.276  0.003   

fate      both          miss               n       
          mean    std   mean    std     mean  std  
arm bin                                            
A0  l    0.000  0.000  0.644  0.072     27.0  0.0  
    m    0.002  0.001  0.166  0.006   3487.0  0.0  
    s    0.006  0.000  0.191  0.003  13761.0  0.0  
    t    0.002  0.000  0.266  0.003  67317.0  0.0  
    vt   0.003  0.000  0.474  0.008  44177.0  0.0  
A2  l    0.022  0.020  0.681  0.050     27.0  0.0  
    m    0.003  0.001  0.177  0.006   3487.0  0.0  
    s    0.005  0.000  0.187  0.003  13761.0  0.0  
    t    0.002  0.000  0.238  0.001  67317.0  0.0  
    vt   0.003  0.000  0.419  0.006  44177.0  0.0

det_0.1        det_0.25        det_0.5        det_0.75       
           mean    std     mean    std    mean    std     mean    std
arm bin                                                              
A0  l     0.348  0.067    0.326  0.066   0.281  0.042    0.163  0.050
    m     0.818  0.006    0.812  0.006   0.773  0.005    0.514  0.006
    s     0.774  0.004    0.762  0.004   0.693  0.005    0.323  0.008
    t     0.689  0.003    0.673  0.003   0.539  0.004    0.132  0.001
    vt    0.464  0.007    0.436  0.007   0.248  0.006    0.022  0.001
A2  l     0.296  0.059    0.267  0.084   0.230  0.048    0.163  0.042
    m     0.806  0.006    0.799  0.006   0.758  0.008    0.497  0.006
    s     0.778  0.002    0.765  0.002   0.692  0.003    0.315  0.007
    t     0.722  0.001    0.705  0.000   0.570  0.005    0.140  0.004
    vt    0.528  0.007    0.498  0.007   0.302  0.005    0.032  0.001

e2e_score0.25 M1_A2-A0_starved_vs_not {'delta': 0.0674, 'lo': 0.0485, 'hi': 0.0862} CI gộp loại 0: True theo seed: {1: 0.0615, 2: 0.0779, 3: 0.064, 4: 0.063, 5: 0.0703}
0.05 A0 1 197s
0.05 A2 1 205s
0.05 A0 2 201s
0.05 A2 2 213s
0.05 A0 3 203s
0.05 A2 3 216s
0.05 A0 4 203s
0.05 A2 4 213s
0.05 A0 5 201s
0.05 A2 5 212s


fate    detected        lost_to_neighbour           cls           loc         \
            mean    std              mean    std   mean    std   mean    std   
arm bin                                                                        
A0  l      0.363  0.031             0.000  0.000  0.000  0.000  0.096  0.033   
    m      0.839  0.007             0.001  0.001  0.002  0.000  0.066  0.005   
    s      0.784  0.004             0.002  0.001  0.004  0.000  0.115  0.005   
    t      0.681  0.006             0.001  0.000  0.003  0.000  0.206  0.004   
    vt     0.392  0.006             0.000  0.000  0.002  0.000  0.393  0.004   
A2  l      0.378  0.055             0.000  0.000  0.000  0.000  0.074  0.026   
    m      0.832  0.004             0.002  0.000  0.002  0.001  0.066  0.002   
    s      0.780  0.004             0.002  0.001  0.004  0.000  0.117  0.002   
    t      0.693  0.008             0.001  0.000  0.003  0.000  0.198  0.006   
    vt     0.453  0.005             0.000  0.000  0.002  0.000  0.355  0.003   

fate      both          miss               n       
          mean    std   mean    std     mean  std  
arm bin                                            
A0  l    0.000  0.000  0.541  0.033     27.0  0.0  
    m    0.007  0.001  0.086  0.004   3487.0  0.0  
    s    0.009  0.000  0.085  0.003  13761.0  0.0  
    t    0.004  0.000  0.104  0.004  67317.0  0.0  
    vt   0.008  0.000  0.206  0.008  44177.0  0.0  
A2  l    0.015  0.020  0.533  0.050     27.0  0.0  
    m    0.006  0.001  0.093  0.003   3487.0  0.0  
    s    0.010  0.001  0.087  0.003  13761.0  0.0  
    t    0.004  0.000  0.101  0.002  67317.0  0.0  
    vt   0.007  0.001  0.182  0.008  44177.0  0.0

det_0.1        det_0.25        det_0.5        det_0.75       
           mean    std     mean    std    mean    std     mean    std
arm bin                                                              
A0  l     0.459  0.033    0.415  0.031   0.363  0.031    0.193  0.061
    m     0.900  0.005    0.888  0.005   0.839  0.007    0.552  0.006
    s     0.892  0.004    0.875  0.004   0.784  0.004    0.358  0.008
    t     0.870  0.004    0.850  0.004   0.681  0.006    0.165  0.003
    vt    0.747  0.008    0.694  0.007   0.392  0.006    0.034  0.001
A2  l     0.452  0.066    0.407  0.037   0.378  0.055    0.259  0.069
    m     0.894  0.004    0.881  0.003   0.832  0.004    0.541  0.006
    s     0.890  0.003    0.873  0.003   0.780  0.004    0.346  0.008
    t     0.876  0.002    0.857  0.003   0.693  0.008    0.166  0.005
    vt    0.784  0.009    0.735  0.008   0.453  0.005    0.047  0.001

e2e_score0.05 M1_A2-A0_starved_vs_not {'delta': 0.1356, 'lo': 0.1097, 'hi': 0.1616} CI gộp loại 0: True theo seed: {1: 0.1306, 2: 0.1343, 3: 0.1405, 4: 0.13, 5: 0.1425}
{
 "p1": {
  "verdict": "Lặp lại được",
  "s4": 0.008508,
  "s5": 0.008728,
  "mean_s4_s5": 0.008618,
  "mei": 0.006,
  "r0_match": false,
  "descriptive_seeds_1to5": {
   "delta": 0.008856049828096284,
   "seed_t": [
    0.003811704744893776,
    0.013900398583582373
   ],
   "scene_ci": [
    0.00524017704680297,
    0.014474774411391491
   ],
   "combined_ci": [
    0.0005960785602883729,
    0.017116021095904196
   ],
   "pooled_allowed": false
  }
 },
 "e2e_score0.25": {
  "M1_A2-A0_starved_vs_not": {
   "delta": 0.06735943963941142,
   "lo": 0.04852947216217686,
   "hi": 0.08618940711664598,
   "supported": true
  }
 },
 "e2e_score0.05": {
  "M1_A2-A0_starved_vs_not": {
   "delta": 0.13560176548446606,
   "lo": 0.1096506223805132,
   "hi": 0.16155290858841892,
   "supported": true
  }
 }
}
XONG. Gửi lại ô này và t